## **Data Preparation**

In [1]:
import pandas as pd
import os
import shutil
import glob
from tqdm import tqdm
from classes import Document
import sys

sys.path.append('..')

### Loading the CSV files and splitting into balanced train, validation, and test

In [2]:
train_original_path = '../data/raw/desegma-it.subTaskA.shared.train.0923-1220.csv'
test_original_path = '../data/raw/desegma-it.subTaskA.with_labels.test.1117_1835.csv'

df_train_full = pd.read_csv(train_original_path, encoding='utf-8')
df_train_0 = df_train_full[df_train_full['label'] == 0]
df_train_1 = df_train_full[df_train_full['label'] == 1]

# Training Set (4000 documents: 2000 per class)
train_0 = df_train_0.sample(n=2000, random_state=42)
train_1 = df_train_1.sample(n=2000, random_state=42)
train_indices = pd.concat([train_0, train_1]).index
train_df = pd.concat([train_0, train_1]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Training set: {len(train_df)} docs")

# Validation Set (2000 documents: 1000 per class)
df_remaining_val = df_train_full.drop(index=train_indices)
val_0 = df_remaining_val[df_remaining_val['label'] == 0].sample(n=1000, random_state=42)
val_1 = df_remaining_val[df_remaining_val['label'] == 1].sample(n=1000, random_state=42)
val_df = pd.concat([val_0, val_1]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Validation set: {len(val_df)} docs")

# Verify overlap before resetting the index
val_indices = pd.concat([val_0, val_1]).index
overlap = len(set(train_indices).intersection(set(val_indices)))
print(f"Overlap between Training and Validation sets: {overlap} docs")

val_df = pd.concat([val_0, val_1]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Validation set: {len(val_df)} docs")

# Test Set (2000 documents: 1000 per class)
df_test_full = pd.read_csv(test_original_path, encoding='utf-8')
test_0 = df_test_full[df_test_full['label'] == 0].sample(n=1000, random_state=42)
test_1 = df_test_full[df_test_full['label'] == 1].sample(n=1000, random_state=42)
test_df = pd.concat([test_0, test_1]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Test set: {len(test_df)} docs")

Training set: 4000 docs
Validation set: 2000 docs
Overlap between Training and Validation sets: 0 docs
Validation set: 2000 docs
Test set: 2000 docs


### Export for Profiling-UD

This section formats the dataset for the ItaliaNLP web portal, which requires individual text files rather than tabular data. This code block implements the following steps:

- Creates a temporary local director to store the files
- Iterates through every row of the Train, Validation and Test sets, saving each text as an independent .txt file
- Applies a specific naming convention (e.g., train_15.txt or val_204.txt) to keep track of the original split and row index for later alignment
- Compresses all the generated .txt files into a single .zip archive, making it ready for a bulk upload to the Profiling-UD tool.


In [3]:
output_dir = 'profiling_upload'

# Clean up the directory if it already exists, then create it
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# Dictionary to easily iterate over the splits
splits = {'train': train_df, 'val': val_df, 'test': test_df}

# Save texts as individual .txt files
for split_name, df in splits.items():
    for idx, row in df.iterrows():
        filename = f"{split_name}_{idx}.txt"
        filepath = os.path.join(output_dir, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(str(row['text']))

# Compress the entire folder into a single .zip archive
# The zip will be created in the same folder where you run the script
shutil.make_archive(output_dir, 'zip', output_dir)
print(f"Export complete: '{output_dir}.zip'.")

# Clean up the unzipped temporary folder after creating the archive
shutil.rmtree(output_dir)

Export complete: 'profiling_upload.zip'.


### Feature Extraction

This section handles the data returned by the Profiling-UD tool, merging it back into the main workflow so it can be used to train the models. This code block implements the followin steps:
- **Lexical Parsing**: Extracts lemmas and POS tags from the downloaded .conllu files
- **Non-Lexical Parsing**: Retrieves the stylistic and syntactic metrics from the Profiling-UD .csv file.  
- **Merging**: Aligns and combines the original texts, labels, and new features using the file IDs
- **Exporting**: Saves the final, enriched datasets as .pkl files to preserve data structures for model training

In [4]:
# Paths where the downloaded Profiling-UD results should be placed
conllu_dir = '../data/profiling_ud/lex_ud'
csv_path = '../data/profiling_ud/nonlex_ud.csv'

# Check if the necessary files have been downloaded from the web portal
if not os.path.exists(conllu_dir) or not os.path.exists(csv_path):
    print("Profiling-UD files not found. Please upload 'profiling_upload.zip' to the ItaliaNLP portal, download the results, and place the CoNLL-U and CSV files in the data folders to continue.")
else:
    conllu_files = glob.glob(os.path.join(conllu_dir, "*.conllu"))
    
    if not conllu_files:
        print("No CoNLL-U files found in the directory.")
    else:
        lexical_data = {'train': [], 'val': [], 'test': []}
        
        # 1. Extract Lexical features (Lemmas, POS)
        for filepath in tqdm(conllu_files, desc="Processing CoNLL-U"):
            filename = os.path.basename(filepath)
            split, id_ext = filename.split('_')
            doc_id = int(id_ext.replace('.conllu', ''))
            
            doc = Document(path=filepath, doc_id=doc_id, split=split, label=None)
            doc.load_sentences_from_conllu() 
            
            lexical_data[split].append({
                'doc_id': doc_id,
                'lemmas_processed': " ".join(doc.get_lemmas()),
                'pos_processed': " ".join(doc.get_pos_tags())
            })

        # 2. Extract Non-Lexical features (CSV)
        df_nonlex = pd.read_csv(csv_path, sep='\t')
        df_nonlex['split'] = df_nonlex['Filename'].apply(lambda x: x.split('_')[0])
        df_nonlex['doc_id'] = df_nonlex['Filename'].apply(lambda x: int(x.split('_')[1].split('.')[0]))
        
        # 3. Merge everything
        out_dir = '../data/enriched'
        os.makedirs(out_dir, exist_ok=True)
        
        for split_name in ['train', 'val', 'test']:
            # Format Lexical Data
            df_lex = pd.DataFrame(lexical_data[split_name]).set_index('doc_id').sort_index()
            
            # Format Non-Lexical Data
            df_nonlex_split = df_nonlex[df_nonlex['split'] == split_name].set_index('doc_id').sort_index()
            df_nonlex_split.drop(columns=['Filename', 'split'], inplace=True)
            
            # Merge with Original Data (Texts and Labels)
            df_original = splits[split_name]
            df_merged = pd.concat([df_original, df_lex, df_nonlex_split], axis=1)
            
            # Save as Pickle to preserve data structures
            save_path = os.path.join(out_dir, f'{split_name}_final.pkl')
            df_merged.to_pickle(save_path)
            
            print(f"Saved {split_name} successfully to {save_path}")

Processing CoNLL-U: 100%|██████████| 8000/8000 [02:39<00:00, 50.20it/s]


Saved train successfully to ../data/enriched\train_final.pkl
Saved val successfully to ../data/enriched\val_final.pkl
Saved test successfully to ../data/enriched\test_final.pkl


In [5]:
# Clean up the ZIP file after the extraction is successfully completed
zip_file = 'profiling_upload.zip'
if os.path.exists(zip_file):
    os.remove(zip_file)
    print(f"Cleaned up temporary file: {zip_file}")

Cleaned up temporary file: profiling_upload.zip
